In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import shap
import joblib
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report, balanced_accuracy_score

warnings.filterwarnings('ignore')

In [ ]:
# 1. Configuration & Data Loading
BASE_PATH = r"D:\Karir\Bootcamp\Data Scientist Rakamin\Week 3\Final Task"
TRAIN_PATH = os.path.join(BASE_PATH, "Code", "train_full_features.csv")
TEST_PATH = os.path.join(BASE_PATH, "Code", "test_full_features.csv")
MODEL_DIR = os.path.join(BASE_PATH, "Model")

if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)

train_full = pd.read_csv(TRAIN_PATH)
test_full = pd.read_csv(TEST_PATH)

X = train_full.drop(columns=['TARGET', 'SK_ID_CURR']).replace([np.inf, -np.inf], np.nan)
y = train_full['TARGET']
test_ids = test_full['SK_ID_CURR']
X_test = test_full.drop(columns=['SK_ID_CURR']).replace([np.inf, -np.inf], np.nan)

In [ ]:
# 2. Categorical Encoding
object_cols = X.select_dtypes(include=['object']).columns
for col in object_cols:
    combined = pd.concat([X[col], X_test[col]], axis=0).astype('category').cat.codes
    X[col] = combined[:len(X)]
    X_test[col] = combined[len(X):]

In [ ]:
# 3. Correlation Filter (> 0.95)
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
X = X.drop(columns=to_drop)
X_test = X_test[X.columns] 

In [ ]:
# 4. Train-Val Split & Ratio
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
imbalance_ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)

In [ ]:
# 5. Hyperparameter Tuning (Optuna)
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 1000, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'scale_pos_weight': imbalance_ratio,
        'eval_metric': 'auc',
        'random_state': 42,
        'tree_method': 'hist'
    }
    model = xgb.XGBClassifier(**params, early_stopping_rounds=50)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=7)

In [ ]:
# 6. Final Model Training
best_params = study.best_params
best_params.update({'scale_pos_weight': imbalance_ratio, 'random_state': 42, 'tree_method': 'hist'})

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100

In [ ]:
# 7. Evaluation & Threshold Tuning
val_probs = final_model.predict_proba(X_val)[:, 1]
thresholds = np.arange(0.1, 0.9, 0.01)
scores = [balanced_accuracy_score(y_val, (val_probs > t).astype(int)) for t in thresholds]
best_threshold = thresholds[np.argmax(scores)]
final_preds = (val_probs > best_threshold).astype(int)

In [ ]:
# 8. Visualizations
# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_val, final_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Lancar', 'Macet'], yticklabels=['Lancar', 'Macet'])
plt.title(f'Confusion Matrix (Threshold: {best_threshold:.2f})')
plt.show()

# SHAP Interpretation (Sampled for speed)
X_shap = X_val.sample(100, random_state=42)
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_shap)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, plot_type="bar")
plt.show()


In [ ]:
# 9. Persistence & Submission
joblib.dump(final_model, os.path.join(MODEL_DIR, 'Model_XGBoost_HCI.pkl'))
X_val.to_pickle(os.path.join(MODEL_DIR, 'X_val_final.pkl'))

test_probs = final_model.predict_proba(X_test)[:, 1]
submission = pd.DataFrame({
    'SK_ID_CURR': test_ids,
    'TARGET_PROB': test_probs,
    'TARGET_PRED': (test_probs > best_threshold).astype(int)
})
submission.to_csv(os.path.join(MODEL_DIR, 'Hasil_Prediksi_Final.csv'), index=False)

print(f"Selesai! Model dan hasil prediksi disimpan di: {MODEL_DIR}")